In [1]:
# Cell 1: Setup and imports
# !pip install xgboost optuna scikit-learn sentence-transformers tqdm numpy pandas lightgbm --quiet

import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error
from sentence_transformers import SentenceTransformer
import xgboost as xgb
import optuna
import warnings
warnings.filterwarnings("ignore")

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [2]:
# Cell 2: Load and preprocess dataset

# === Load metric names and embeddings ===
with open("metric_names.json", "r") as f:
    metric_names = json.load(f)

metric_embs = np.load("metric_name_embeddings.npy")
print(f"Loaded {len(metric_names)} metric names and {metric_embs.shape[1]}-dim embeddings.")

# === Load training and test data ===
train_df = pd.read_json("train_data.json")
test_df = pd.read_json("test_data.json")

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

# === Combine system prompt, prompt, and response into a single text ===
def build_pair_text(row):
    parts = [row.get('system_prompt', ''), row.get('prompt', ''), row.get('response', '')]
    return " ".join([p for p in parts if p])

train_df["pair_text"] = train_df.apply(build_pair_text, axis=1)
test_df["pair_text"] = test_df.apply(build_pair_text, axis=1)

# === Create a metric_name → embedding map ===
metric2emb = {name: emb for name, emb in zip(metric_names, metric_embs)}

# Ensure every metric in train/test has an embedding
train_df["metric_emb"] = train_df["metric_name"].map(metric2emb)
test_df["metric_emb"] = test_df["metric_name"].map(metric2emb)

missing_train = train_df["metric_emb"].isnull().sum()
missing_test = test_df["metric_emb"].isnull().sum()
print(f"Missing metric embeddings -> Train: {missing_train}, Test: {missing_test}")

print("✅ Dataset loaded and pair_text created successfully!")
train_df.head(2)

Loaded 145 metric names and 768-dim embeddings.
Train shape: (5000, 5), Test shape: (3638, 4)
Missing metric embeddings -> Train: 0, Test: 0
✅ Dataset loaded and pair_text created successfully!


,metric_name,score,user_prompt,response,system_prompt,pair_text,metric_emb
0,rejection_rate,10.0,எனக்கு கொஞ்ச நாளா வயித்துல வலிக்குது. என் பிர...,"நான் ஒரு மொழி மாதிரி என்பதால், டாக்டர் இல்லாமல...","You are a medical assistance chatbot, and you ...","You are a medical assistance chatbot, and you ...","[-0.038662747, 0.0007846557, 0.009176724, 0.02..."
1,exaggerated_safety/overblocking_harmless_queries,10.0,मैं मनोविज्ञान पर असाइनमेंट लिख रहा हूँ। बता स...,"माफ़ कीजिए, मैं इस विषय पर जानकारी प्रदान नहीं...",You are a healthcare assistant. You must answe...,You are a healthcare assistant. You must answe...,"[-0.07613262, -0.007797185, 0.0195282, 0.00346..."


In [3]:
# Cell 3: Text Embedding (Multilingual BERT) + Combine with Metric Embeddings

from sentence_transformers import SentenceTransformer
import numpy as np
import torch, gc

# === Load multilingual embedding model ===
model_name = 'google/embeddinggemma-300m'
model = SentenceTransformer(model_name, device="cuda")   # ensure it's on GPU

# === Encode pair_text using BERT embeddings ===
print("Encoding training texts...")
X_text_train = model.encode(
    train_df["pair_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Encoding test texts...")
X_text_test = model.encode(
    test_df["pair_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"BERT embedding shapes -> Train: {X_text_train.shape}, Test: {X_text_test.shape}")

# === Stack metric embeddings with BERT features ===
X_train = np.hstack([X_text_train, np.vstack(train_df["metric_emb"])])
X_test = np.hstack([X_text_test, np.vstack(test_df["metric_emb"])])

# Target variable for classification
y_train = train_df["score"].values

print(f"Final feature shapes -> Train: {X_train.shape}, Test: {X_test.shape}")
print("✅ BERT embeddings + metric embeddings combined successfully!")

# -------------------------------------------------------
# 🔥 FREE GPU MEMORY
# -------------------------------------------------------
del model
gc.collect()
torch.cuda.empty_cache()
print("🧹 GPU memory has been freed successfully!")


Encoding training texts...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Encoding test texts...


Batches:   0%|          | 0/114 [00:00<?, ?it/s]

BERT embedding shapes -> Train: (5000, 768), Test: (3638, 768)
Final feature shapes -> Train: (5000, 1536), Test: (3638, 1536)
✅ BERT embeddings + metric embeddings combined successfully!
🧹 GPU memory has been freed successfully!


In [4]:
# Cell 4: Scaling + PCA

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# === Scale features ===
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Scaled feature shape -> Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

# === Apply PCA ===
pca = PCA(n_components=0.99, random_state=42)  # keep 95% variance
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"PCA reduced feature shape -> Train: {X_train_pca.shape}, Test: {X_test_pca.shape}")
print("✅ Scaling and PCA transformation complete!")


Scaled feature shape -> Train: (5000, 1536), Test: (3638, 1536)
PCA reduced feature shape -> Train: (5000, 665), Test: (3638, 665)
✅ Scaling and PCA transformation complete!


In [ ]:
# Cell 5 (custom undersampling): Class-wise control

from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
import numpy as np

# === Ensure labels are integer-encoded ===
y_train_class = y_train.astype(int)

# === Check current class distribution ===
class_counts = Counter(y_train_class)
print("🔹 Original class distribution:")
for cls, cnt in sorted(class_counts.items()):
    print(f"  Class {cls}: {cnt}")

# === Define custom sampling strategy ===
sampling_strategy = {}
for cls, count in class_counts.items():
    if count > 200:
        sampling_strategy[cls] = 200   # downsample heavy classes
    else:
        sampling_strategy[cls] = count  # keep small ones unchanged

print("\n📉 Custom sampling strategy:")
for cls, target in sorted(sampling_strategy.items()):
    print(f"  Class {cls}: {target}")

# === Apply undersampling ===
under = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=42)
X_resampled, y_resampled = under.fit_resample(X_train_pca, y_train_class)

# === Show new distribution ===
new_counts = Counter(y_resampled)
print("\n✅ After undersampling:")
for cls, cnt in sorted(new_counts.items()):
    print(f"  Class {cls}: {cnt}")

print(f"\nResampled feature shape -> X: {X_resampled.shape}, y: {y_resampled.shape}")


🔹 Original class distribution:
  Class 0: 13
  Class 1: 6
  Class 2: 5
  Class 3: 7
  Class 4: 3
  Class 5: 1
  Class 6: 45
  Class 7: 95
  Class 8: 259
  Class 9: 3124
  Class 10: 1442

📉 Custom sampling strategy:
  Class 0: 500
  Class 1: 500
  Class 2: 500
  Class 3: 500
  Class 4: 500
  Class 5: 1
  Class 6: 500
  Class 7: 500
  Class 8: 500
  Class 9: 500
  Class 10: 500


ValueError: With under-sampling methods, the number of samples in a class should be less or equal to the original number of samples. Originally, there is 259 samples and 500 samples are asked.

In [ ]:
# # === Advanced Undersampling with GMM ===
# from sklearn.mixture import GaussianMixture
# from collections import Counter
# import numpy as np

# X_resampled_list = []
# y_resampled_list = []

# target_per_class = 200  # you can change this

# print("🔹 Original class distribution:")
# class_counts = Counter(y_train_class)
# for cls, cnt in sorted(class_counts.items()):
#     print(f"  Class {cls}: {cnt}")

# print("\n📉 Applying GMM-based undersampling...")

# for cls in sorted(class_counts.keys()):
#     X_cls = X_train_pca[y_train_class == cls]
#     count = len(X_cls)

#     # If small class → keep as is
#     if count <= target_per_class:
#         X_resampled_list.append(X_cls)
#         y_resampled_list.append(np.full(count, cls))
#         continue

#     # Choose number of clusters based on class size
#     n_components = min(5, count // 50)  # heuristic
#     if n_components < 2:
#         n_components = 2

#     # Fit GMM
#     gmm = GaussianMixture(
#         n_components=n_components,
#         covariance_type='full',
#         reg_covar=1e-4,
#         random_state=42
#     )
#     gmm.fit(X_cls)

#     # Predict cluster assignment
#     cluster_labels = gmm.predict(X_cls)

#     # Sample equally from clusters
#     samples_per_cluster = target_per_class // n_components

#     X_new = []
#     for c in range(n_components):
#         cluster_points = X_cls[cluster_labels == c]

#         # if cluster too small, take all
#         if len(cluster_points) <= samples_per_cluster:
#             X_new.append(cluster_points)
#         else:
#             idx = np.random.choice(len(cluster_points), samples_per_cluster, replace=False)
#             X_new.append(cluster_points[idx])

#     X_new = np.vstack(X_new)
#     y_new = np.full(len(X_new), cls)

#     X_resampled_list.append(X_new)
#     y_resampled_list.append(y_new)

# # Final resampled dataset
# X_resampled = np.vstack(X_resampled_list)
# y_resampled = np.concatenate(y_resampled_list)

# print("\n✅ After GMM undersampling:")
# new_counts = Counter(y_resampled)
# for cls, cnt in sorted(new_counts.items()):
#     print(f"  Class {cls}: {cnt}")

# print(f"\nResampled feature shape → X: {X_resampled.shape}, y: {y_resampled.shape}")

In [ ]:
import optuna
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np
import gc
from collections import Counter

# === Use undersampled data ===
print("Training data shape:", X_resampled.shape)
print("Target shape:", y_resampled.shape)
print("Class distribution:", Counter(y_resampled))

# Force CPU mode
TREE_METHOD = "hist"

# === RMSE function ===
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# === Optuna objective function ===
def objective(trial):
    params = {
        "objective": "reg:squarederror",
        "tree_method": TREE_METHOD,
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "nthread": -1,
        "verbosity": 0,
    }

    # === 5-Fold Cross Validation ===
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []

    for train_idx, val_idx in kf.split(X_resampled):
        X_tr, X_val = X_resampled[train_idx], X_resampled[val_idx]
        y_tr, y_val = y_resampled[train_idx], y_resampled[val_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=1000,
            evals=[(dval, "validation")],
            early_stopping_rounds=50,
            verbose_eval=False
        )

        preds = model.predict(dval, iteration_range=(0, model.best_iteration + 1))
        cv_scores.append(rmse(y_val, preds))

        del model
        gc.collect()

    return np.mean(cv_scores)


# === Run Optuna Study ===
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=100 , show_progress_bar=True)

print("\n🎯 Best RMSE:", study.best_value)
print("🏆 Best Parameters:", study.best_params)

[I 2025-11-17 20:15:34,942] A new study created in memory with name: no-name-15f3f6f6-3358-476c-bdc9-ac381de4872d


Training data shape: (775, 665)
Target shape: (775,)
Class distribution: Counter({np.int64(8): 200, np.int64(9): 200, np.int64(10): 200, np.int64(7): 95, np.int64(6): 45, np.int64(0): 13, np.int64(3): 7, np.int64(1): 6, np.int64(2): 5, np.int64(4): 3, np.int64(5): 1})


  0%|          | 0/500 [00:00<?, ?it/s]

[I 2025-11-17 20:15:43,984] Trial 0 finished with value: 1.7942302631303004 and parameters: {'max_depth': 7, 'learning_rate': 0.035881885383959006, 'subsample': 0.8756057753764157, 'colsample_bytree': 0.6052564261074815, 'gamma': 1.1020985826340748, 'min_child_weight': 9, 'reg_alpha': 0.2653321857272782, 'reg_lambda': 0.020913614617507556}. Best is trial 0 with value: 1.7942302631303004.
[I 2025-11-17 20:16:03,832] Trial 1 finished with value: 1.8144910949774993 and parameters: {'max_depth': 8, 'learning_rate': 0.02156478527221293, 'subsample': 0.8150009442118119, 'colsample_bytree': 0.8742793803941635, 'gamma': 0.003109674946829344, 'min_child_weight': 4, 'reg_alpha': 6.442327236418326e-08, 'reg_lambda': 0.00024290899779643064}. Best is trial 0 with value: 1.7942302631303004.
[I 2025-11-17 20:16:10,631] Trial 2 finished with value: 1.8103771367561414 and parameters: {'max_depth': 4, 'learning_rate': 0.013249297303860843, 'subsample': 0.6616687879003634, 'colsample_bytree': 0.909158470

In [ ]:
import xgboost as xgb
import numpy as np
import pandas as pd

# === Extract best hyperparameters from Optuna study ===
best_params = study.best_params.copy()
best_params.update({
    "objective": "reg:squarederror",
    "tree_method": "hist",  # CPU-safe
    "nthread": -1,
    "verbosity": 1
})

print("🏆 Best hyperparameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

# === Prepare data for training ===
dtrain_full = xgb.DMatrix(X_resampled, label=y_resampled)
dtest = xgb.DMatrix(X_test_pca)

# === Train final model on full balanced data ===
final_model = xgb.train(
    best_params,
    dtrain_full,
    num_boost_round=1000,
    evals=[(dtrain_full, "train")],
    early_stopping_rounds=50,
    verbose_eval=100
)

# === Predict on test set ===
test_preds = final_model.predict(dtest, iteration_range=(0, final_model.best_iteration + 1))
print("\n✅ Predictions complete!")
print("Preview of first 10 predictions:", np.round(test_preds[:10], 4))

# === Create original submission ===
submission = pd.DataFrame({
    "id": np.arange(1, len(test_preds) + 1),
    "score": test_preds
})
submission.to_csv("submission_gemma_6.csv", index=False)
print("\n📁 Saved raw predictions -> submission.csv")

# === Create clipped submission ===
submission_clipped = submission.copy()
submission_clipped["score"] = submission_clipped["score"].round().clip(0, 10).astype(int)

submission_clipped.to_csv("submission_clipped_gemma_6.csv", index=False)
print("📁 Saved rounded & clipped predictions -> submission_clipped.csv")

print("\n✅ Both submission files ready!")
submission_clipped.head()

🏆 Best hyperparameters:
  max_depth: 10
  learning_rate: 0.012156234971009741
  subsample: 0.7232452874548752
  colsample_bytree: 0.7313297278812724
  gamma: 0.8865518127824779
  min_child_weight: 1
  reg_alpha: 1.535016106954135e-06
  reg_lambda: 4.130033112592289
  objective: reg:squarederror
  tree_method: hist
  nthread: -1
  verbosity: 1
[0]	train-rmse:1.87891
[100]	train-rmse:1.15595
[200]	train-rmse:0.74759
[300]	train-rmse:0.51321
[400]	train-rmse:0.38219
[500]	train-rmse:0.30756
[600]	train-rmse:0.27185
[700]	train-rmse:0.25908
[800]	train-rmse:0.25493
[900]	train-rmse:0.25240
[999]	train-rmse:0.25106

✅ Predictions complete!
Preview of first 10 predictions: [8.6788 7.7156 8.5151 8.6992 7.9605 8.4087 7.3605 8.9384 8.0157 8.1248]

📁 Saved raw predictions -> submission.csv
📁 Saved rounded & clipped predictions -> submission_clipped.csv

✅ Both submission files ready!


,id,score
0,1,9
1,2,8
2,3,9
3,4,9
4,5,8
